In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# Load daily returns from silver table
df = spark.sql("SELECT Date, Ticker, Sector, Close, daily_return_pct FROM silver_stock_analytics ORDER BY Ticker, Date").toPandas()

# Pivot to wide format: each column is a stock, each row is a day
returns = df.pivot_table(index="Date", columns="Ticker", values="daily_return_pct")

# Drop any columns with missing values
returns = returns.dropna(axis=1)

print(f"Stocks: {returns.shape[1]}")
print(f"Trading days: {returns.shape[0]}")
print(f"\nAverage daily return per stock:")
print(returns.mean().sort_values(ascending=False).head(10))

Stocks: 50
Trading days: 1271

Average daily return per stock:
Ticker
NVDA     0.249650
AMD      0.207350
LLY      0.160844
GE       0.150583
CAT      0.136819
TSLA     0.114775
GS       0.106172
GOOGL    0.103837
MS       0.097417
XOM      0.092999
dtype: float64


In [0]:
# PORTFOLIO OPTIMIZATION

# Calculate key inputs
mean_returns = returns.mean()                    # average daily return per stock
cov_matrix = returns.cov()                       # how stocks move together
num_stocks = len(mean_returns)

print(f"Covariance matrix shape: {cov_matrix.shape}")
print(f"Stocks in portfolio: {num_stocks}")

# ============================================
# STRATEGY 1: MINIMUM RISK PORTFOLIO
# "Give me the safest possible portfolio"
# ============================================

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

# Constraints: weights must add up to 1 (100% invested)
constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}

# Bounds: each stock gets between 0% and 15% (no shorting, no single stock dominates)
bounds = tuple((0, 0.15) for _ in range(num_stocks))

# Starting point: equal weight
initial_weights = np.array([1/num_stocks] * num_stocks)

# Find the weights that minimize risk
min_risk = minimize(portfolio_volatility, initial_weights, args=(cov_matrix,),
                    method="SLSQP", bounds=bounds, constraints=constraints)

min_risk_weights = min_risk.x
min_risk_return = np.dot(min_risk_weights, mean_returns)
min_risk_vol = portfolio_volatility(min_risk_weights, cov_matrix)

print("\n" + "=" * 50)
print("STRATEGY 1: MINIMUM RISK PORTFOLIO")
print("=" * 50)
print(f"Expected daily return: {min_risk_return:.4f}%")
print(f"Daily volatility:      {min_risk_vol:.4f}%")
print(f"Sharpe proxy:          {min_risk_return/min_risk_vol:.4f}")
print(f"\nTop 10 allocations:")
min_risk_alloc = pd.DataFrame({"Ticker": mean_returns.index, "Weight": min_risk_weights})
min_risk_alloc = min_risk_alloc[min_risk_alloc["Weight"] > 0.001].sort_values("Weight", ascending=False)
print(min_risk_alloc.head(10).to_string(index=False))

# ============================================
# STRATEGY 2: MAXIMUM SHARPE PORTFOLIO
# "Give me the best return for each unit of risk"
# ============================================

def neg_sharpe(weights, mean_returns, cov_matrix):
    port_return = np.dot(weights, mean_returns)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -port_return / port_vol  # negative because we minimize

max_sharpe = minimize(neg_sharpe, initial_weights, args=(mean_returns, cov_matrix),
                      method="SLSQP", bounds=bounds, constraints=constraints)

max_sharpe_weights = max_sharpe.x
max_sharpe_return = np.dot(max_sharpe_weights, mean_returns)
max_sharpe_vol = portfolio_volatility(max_sharpe_weights, cov_matrix)

print("\n" + "=" * 50)
print("STRATEGY 2: MAXIMUM SHARPE PORTFOLIO")
print("=" * 50)
print(f"Expected daily return: {max_sharpe_return:.4f}%")
print(f"Daily volatility:      {max_sharpe_vol:.4f}%")
print(f"Sharpe proxy:          {max_sharpe_return/max_sharpe_vol:.4f}")
print(f"\nTop 10 allocations:")
max_sharpe_alloc = pd.DataFrame({"Ticker": mean_returns.index, "Weight": max_sharpe_weights})
max_sharpe_alloc = max_sharpe_alloc[max_sharpe_alloc["Weight"] > 0.001].sort_values("Weight", ascending=False)
print(max_sharpe_alloc.head(10).to_string(index=False))

# ============================================
# STRATEGY 3: EQUAL WEIGHT (BASELINE)
# "Just split evenly across all stocks"
# ============================================

equal_weights = np.array([1/num_stocks] * num_stocks)
equal_return = np.dot(equal_weights, mean_returns)
equal_vol = portfolio_volatility(equal_weights, cov_matrix)

print("\n" + "=" * 50)
print("STRATEGY 3: EQUAL WEIGHT (BASELINE)")
print("=" * 50)
print(f"Expected daily return: {equal_return:.4f}%")
print(f"Daily volatility:      {equal_vol:.4f}%")
print(f"Sharpe proxy:          {equal_return/equal_vol:.4f}")

# ============================================
# COMPARISON
# ============================================
print("\n" + "=" * 50)
print("STRATEGY COMPARISON")
print("=" * 50)
comparison = pd.DataFrame({
    "Strategy": ["Min Risk", "Max Sharpe", "Equal Weight"],
    "Daily Return (%)": [min_risk_return, max_sharpe_return, equal_return],
    "Daily Volatility (%)": [min_risk_vol, max_sharpe_vol, equal_vol],
    "Sharpe Proxy": [min_risk_return/min_risk_vol, max_sharpe_return/max_sharpe_vol, equal_return/equal_vol]
}).round(4)
print(comparison.to_string(index=False))

Covariance matrix shape: (50, 50)
Stocks in portfolio: 50

STRATEGY 1: MINIMUM RISK PORTFOLIO
Expected daily return: 0.0519%
Daily volatility:      0.7163%
Sharpe proxy:          0.0724

Top 10 allocations:
Ticker   Weight
   JNJ 0.150000
    KO 0.127423
   MCD 0.108059
   LMT 0.095353
    PG 0.086188
  MSFT 0.084315
   WMT 0.077056
   CVX 0.048925
   XOM 0.044570
   MRK 0.037522

STRATEGY 2: MAXIMUM SHARPE PORTFOLIO
Expected daily return: 0.1218%
Daily volatility:      0.9720%
Sharpe proxy:          0.1253

Top 10 allocations:
Ticker   Weight
   WMT 0.150000
   LLY 0.150000
   XOM 0.143940
  NVDA 0.125874
  ABBV 0.114831
    GE 0.098161
   JNJ 0.081579
    KO 0.069611
   CAT 0.057654
   MRK 0.008351

STRATEGY 3: EQUAL WEIGHT (BASELINE)
Expected daily return: 0.0645%
Daily volatility:      0.9965%
Sharpe proxy:          0.0647

STRATEGY COMPARISON
    Strategy  Daily Return (%)  Daily Volatility (%)  Sharpe Proxy
    Min Risk            0.0519                0.7163        0.0724
  Max 

In [0]:
# Save portfolio optimization results

# Save strategy comparison
strategy_df = pd.DataFrame({
    "Strategy": ["Min Risk", "Max Sharpe", "Equal Weight"],
    "Daily_Return_Pct": [min_risk_return, max_sharpe_return, equal_return],
    "Daily_Volatility_Pct": [min_risk_vol, max_sharpe_vol, equal_vol],
    "Sharpe_Proxy": [min_risk_return/min_risk_vol, max_sharpe_return/max_sharpe_vol, equal_return/equal_vol]
})
spark.createDataFrame(strategy_df).write.mode("overwrite").saveAsTable("gold_portfolio_strategies")

# Save all stock allocations for both optimized strategies
min_alloc = pd.DataFrame({"Ticker": mean_returns.index, "Weight": min_risk_weights, "Strategy": "Min Risk"})
max_alloc = pd.DataFrame({"Ticker": mean_returns.index, "Weight": max_sharpe_weights, "Strategy": "Max Sharpe"})
equal_alloc = pd.DataFrame({"Ticker": mean_returns.index, "Weight": equal_weights, "Strategy": "Equal Weight"})

all_allocations = pd.concat([min_alloc, max_alloc, equal_alloc])
spark.createDataFrame(all_allocations).write.mode("overwrite").saveAsTable("gold_portfolio_allocations")

# Simulate portfolio performance over time (backtest)
# Apply each strategy's weights to actual daily returns
returns_array = returns.values

min_risk_daily = returns_array @ min_risk_weights
max_sharpe_daily = returns_array @ max_sharpe_weights
equal_daily = returns_array @ equal_weights

# Convert to cumulative growth ($100 invested)
dates = returns.index
backtest = pd.DataFrame({
    "Date": dates,
    "Min_Risk": (1 + min_risk_daily/100).cumprod() * 100,
    "Max_Sharpe": (1 + max_sharpe_daily/100).cumprod() * 100,
    "Equal_Weight": (1 + equal_daily/100).cumprod() * 100
})

spark.createDataFrame(backtest).write.mode("overwrite").saveAsTable("gold_portfolio_backtest")

print("All portfolio tables saved!")
print(f"\n$100 invested at start would be worth today:")
print(f"  Min Risk:     ${backtest['Min_Risk'].iloc[-1]:.2f}")
print(f"  Max Sharpe:   ${backtest['Max_Sharpe'].iloc[-1]:.2f}")
print(f"  Equal Weight: ${backtest['Equal_Weight'].iloc[-1]:.2f}")

All portfolio tables saved!

$100 invested at start would be worth today:
  Min Risk:     $187.16
  Max Sharpe:   $442.57
  Equal Weight: $213.03
